# This notebook is used to build a Decision Tree model using the Diabetes Readmission dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## These are some of the prerequisites to run this notebook when running in the community ediction of databricks

1. Import the diabetes_readmit_logreg_class1_csv dataset file from your system


In [2]:
# Install Java Development Kit (JDK)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [3]:
# Install PySpark
!pip install pyspark==3.3.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.3/281.3 MB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 11.4 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.3.0-py2.py3-none-any.whl size=281764007 sha256=a0cbd8dd159b2978d21f01af6d2005b21d88f3b279528966eb1b5e598cfe4acd
  Stored in directory: /root/.cache/pip/wheels/fa/38/b6/02a8fa3ddb890ded2545483c78766bb378d8f505dc4f26d917
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [4]:
# Set environment variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"

In [5]:
# Initialize SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .appName("MedicalInsuranceRegression") \
    .getOrCreate()
print("SparkSession initialized.")

SparkSession initialized.


In [18]:
from pyspark.sql.functions import log, col, exp

# Modeling
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [10]:
diabetes_readmit = spark.read.csv('/content/drive/MyDrive/UofLClasswork/Datasets/diab_readmit_uofl.csv', header=True, inferSchema=True)

#Show basic summary stats
display(diabetes_readmit.summary().toPandas())

,summary,patient_nbr,time_in_hospital,num_procedures,num_lab_procedures,num_medications,number_outpatient,number_inpatient,number_emergency,number_diagnoses,gender_cd,DiabetesMedication,readmit_flag,race_cd
0,count,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766,101766
1,mean,5.4330400694947235E7,4.395986871843248,1.339730361810428,43.09564098028811,16.021844230882614,0.36935715268360747,0.635565906098304,0.19783621248747127,7.422606764538254,0.5375862272271682,0.7700312481575379,0.11159915885462728,None
2,stddev,3.869635934653421E7,2.985107767471267,1.705806979121172,19.674362249142096,8.127566209167309,1.2672650965326817,1.26286329009732,0.9304722684224632,1.9336001449974247,0.49858772375671534,0.420814525814695,0.3148741984505526,None
3,min,135,1,0,1,1,0,0,0,1,0,0,0,AfrAmr
4,25%,23412645,2,0,31,10,0,0,0,6,0,1,0,None
5,50%,45500490,4,1,44,15,0,0,0,8,1,1,0,None
6,75%,87532902,6,2,57,20,0,1,0,9,1,1,0,None
7,max,189502619,14,6,132,81,42,21,76,16,1,1,1,White


In [11]:
# Train test split
trainDF, testDF = diabetes_readmit.randomSplit([.65, .35], seed=42)
# Print the number of records
print(f'There are {trainDF.cache().count()} records in the training dataset.')
print(f'There are {testDF.cache().count()} records in the testing dataset.')

There are 66284 records in the training dataset.
There are 35482 records in the testing dataset.


In [12]:
#One hot encoding of string feature Region
trainDF.groupBy('race_cd').count().show()

+--------+-----+
| race_cd|count|
+--------+-----+
|     Unk| 2472|
|  AfrAmr|12558|
|   White|49512|
|Hispanic| 1326|
|   Asian|  416|
+--------+-----+



##Now we need to modify the categorical variable race_cd into one-hot-encoded version
 For this we will use a pipeline to do this in one step

In [13]:
#You can also create a pipeline and do everything together in one easy fit and transform step
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

categoricalColumns = ["race_cd"]
stages = [] # stages in Pipeline
for categoricalCol in categoricalColumns:
    # Category Indexing with StringIndexer
    stringIndexer = StringIndexer(inputCol=categoricalCol, outputCol=categoricalCol + "Index")

# Use OneHotEncoder to convert categorical variables into binary SparseVectors
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder(inputCols=[stringIndexer.getOutputCol()], outputCols=[categoricalCol + "classVec"])

# Define the pipeline based on the stages created in previous steps.
pipeline = Pipeline(stages=[stringIndexer, encoder])

# Define the pipeline model.
transform_mdl = pipeline.fit(trainDF)
trainDF21=transform_mdl.transform(trainDF)
trainDF21.show()

+-----------+----------------+--------------+------------------+---------------+-----------------+----------------+----------------+----------------+---------+------------------+------------+-------+------------+---------------+
|patient_nbr|time_in_hospital|num_procedures|num_lab_procedures|num_medications|number_outpatient|number_inpatient|number_emergency|number_diagnoses|gender_cd|DiabetesMedication|readmit_flag|race_cd|race_cdIndex|race_cdclassVec|
+-----------+----------------+--------------+------------------+---------------+-----------------+----------------+----------------+----------------+---------+------------------+------------+-------+------------+---------------+
|        135|               3|             1|                31|             14|                0|               1|               0|               5|        1|                 1|           0|  White|         0.0|  (4,[0],[1.0])|
|        135|               8|             6|                77|             33|    

In [14]:
trainDF21.printSchema()

root
 |-- patient_nbr: integer (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- number_outpatient: integer (nullable = true)
 |-- number_inpatient: integer (nullable = true)
 |-- number_emergency: integer (nullable = true)
 |-- number_diagnoses: integer (nullable = true)
 |-- gender_cd: integer (nullable = true)
 |-- DiabetesMedication: integer (nullable = true)
 |-- readmit_flag: integer (nullable = true)
 |-- race_cd: string (nullable = true)
 |-- race_cdIndex: double (nullable = false)
 |-- race_cdclassVec: vector (nullable = true)



In [15]:
# Linear regression expect a vector input
vecAssembler = VectorAssembler(inputCols=['time_in_hospital','num_procedures','num_medications', 'number_inpatient','number_emergency','number_diagnoses','DiabetesMedication'], outputCol="features")
vecTrainDF = vecAssembler.transform(trainDF21)

In [16]:
testDF21=transform_mdl.transform(testDF) #do the data transformation using saved parameters from training
vecTestDF = vecAssembler.transform(testDF21) #do the feature transformation using vector assembler

In [19]:
# Create Decision tree calssifier
dt = DecisionTreeClassifier(featuresCol="features", labelCol="readmit_flag", maxDepth = 20)
dtModel = dt.fit(vecTrainDF)
predict_train = dtModel.transform(vecTrainDF)
predict_train.select('readmit_flag', 'rawPrediction', 'prediction', 'probability').show(10)

+------------+-------------+----------+--------------------+
|readmit_flag|rawPrediction|prediction|         probability|
+------------+-------------+----------+--------------------+
|           0|    [9.0,0.0]|       0.0|           [1.0,0.0]|
|           1| [405.0,35.0]|       0.0|[0.92045454545454...|
|           0| [262.0,27.0]|       0.0|[0.90657439446366...|
|           0|   [81.0,1.0]|       0.0|[0.98780487804878...|
|           0|   [30.0,0.0]|       0.0|           [1.0,0.0]|
|           0|   [22.0,0.0]|       0.0|           [1.0,0.0]|
|           1|    [0.0,1.0]|       1.0|           [0.0,1.0]|
|           0|   [58.0,2.0]|       0.0|[0.96666666666666...|
|           0| [405.0,35.0]|       0.0|[0.92045454545454...|
|           0| [356.0,10.0]|       0.0|[0.97267759562841...|
+------------+-------------+----------+--------------------+
only showing top 10 rows



In [20]:
# Make predictions on testing dataset
predict_test = dtModel.transform(vecTestDF) #make predictions using the trained model
predict_test.groupBy('prediction').count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       0.0|33924|
|       1.0| 1558|
+----------+-----+



In [21]:
eval = BinaryClassificationEvaluator(rawPredictionCol = "prediction", labelCol = "readmit_flag")
auc_train = eval.evaluate(predict_train)
print(auc_train)

auc_test = eval.evaluate(predict_test)
print(auc_test)

0.6455949040237675
0.5150044336259191


## Assignment 2.1

1. Create a Decision Tree model with Max Depth 15, and 25 each
1. Which model has the highest test AUC?